<a href="https://colab.research.google.com/github/swarnapatro/AI_KickStart/blob/main/mcp_in_6_cells_ollama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MCP in 6 cells — how the Model Context Protocol actually works
### (Free version: runs on Google Colab with a free local LLM from **Ollama** — no API key, no account, no cost)

### The one-sentence version

> **MCP is a way for a program that has tools to tell any AI app what those tools are — over
> plain JSON messages sent through a pipe.**

That is genuinely all it is. No magic, no framework required. This notebook proves it by talking
to a real MCP server with 15 lines of ordinary Python and no MCP library at all.

---

### How to use this notebook (read this first!)

1. Open it in **Google Colab** (colab.research.google.com → File → Upload notebook).
2. *(Recommended)* Turn on the free GPU: **Runtime → Change runtime type → T4 GPU → Save**.
   It works without a GPU too, just slower in Cell 5.
3. Run the cells **one at a time, top to bottom**: click a cell, press **Shift + Enter**.
4. Wait for each cell to finish (the little ▶ icon stops spinning) before running the next.
5. **You do not need to understand every line of Python.** Each cell has a plain-English
   explanation above it, and comments (lines starting with `#`) inside it. Read those first.

> **What is a "comment"?** A line starting with `#` is a note for humans. Python ignores it.
> We use lots of them here.

---

### The problem MCP solves

Before MCP, if you wanted an AI app to read your orders database you wrote glue code — for *that*
app. Wanted it in a second app too? Write the glue again, in that app's format.

*N* apps × *M* data sources = *N × M* integrations.

MCP makes it *N + M*: a data source implements the protocol **once** as a **server**, and every
AI app that speaks MCP can use it. It is the USB-C analogy people use, and for once the analogy
is accurate: one plug shape, any device.

---

### The three roles

| Role | What it is | In this notebook |
|---|---|---|
| **Server** | A program that owns some capability and describes it in MCP's format | `orders_server.py` — knows about orders |
| **Client** | The plumbing that speaks the protocol to the server | Cell 4 (raw) and Cell 6 (SDK) |
| **Host** | The AI app the human talks to; it embeds a client and lets the LLM choose | Cell 5 — here the "brain" is a free **Ollama** model |

---

### What is Ollama?

**Ollama** is a free program that downloads open-source AI models (Llama, Qwen, Gemma, …) and
runs them on *your own machine* — here, on the free Colab computer. No internet API, no key,
no bill. We use the small model **`llama3.2:3b`** (about 2 GB) because it is fast and it supports
**tool calling** — the ability to say *"I would like to use this tool"* — which MCP needs.

---

### What we build

| Cell | What it does |
|---|---|
| 1 | Install the Python libraries |
| 2 | **Set up Ollama** — install it, start it, download the free model |
| 3 | **The server** — 12 lines, one tool |
| 4 | **The protocol, naked** — raw JSON over a pipe, no MCP library |
| 5 | **End to end** — the Ollama LLM decides to call the tool and answers the human |
| 6 | **The version you would actually ship** — the official SDK, 8 lines |

Cells 3, 4 and 6 need no LLM at all; they show MCP itself.

### Cell 1 — Install the Python libraries

**What it does:** installs two helper libraries (think of them as "apps for Python"):

| Library | Why we need it |
|---|---|
| `mcp` | The official MCP Python SDK. We use it to **build the server** (Cell 3) and as a ready-made **client** (Cell 6). |
| `ollama` | The official Python library for talking to the Ollama LLM (Cells 2 and 5). |

**Input:** nothing — just run it.

**Output:** a few install messages, then something like:
```
mcp version: 1.30.0
Server API : FastMCP (mcp 1.x)
Done. If this was your first install, restart the kernel now.
```

> **Which SDK version am I on?** The `mcp` SDK had a breaking rename at **2.0**:
> `mcp.server.fastmcp.FastMCP` became `mcp.server.mcpserver.MCPServer`. This notebook
> detects the version and works on both, so you do not have to care — but the printout
> tells you which one you are running, which is worth knowing when you read other tutorials.
>
> **If Colab asks you to "Restart session"** after the install, click it, then continue from Cell 2
> (you do not need to re-run Cell 1).

In [ ]:
# %pip is a Jupyter/Colab command (not normal Python). It installs libraries into THIS notebook.
# "-q" = quiet: hide the long install log.
%pip install -q mcp ollama

# ---------- IMPORT ----------
# importlib.metadata -> a built-in Python tool that can ask "which version of library X is installed?"
# We use it ONLY to print the installed `mcp` version, because the server code differs slightly
# between version 1 and version 2.
from importlib.metadata import version

try:
    mcp_version = version("mcp")          # e.g. "1.30.0" or "2.0.1"
except Exception:
    mcp_version = "unknown"               # if the lookup fails, we simply say "unknown"

print("mcp version:", mcp_version)
# mcp_version[:1] is the first character ("1" or "2") -> tells us which server class exists.
print("Server API :", "MCPServer (mcp 2.x)" if mcp_version[:1] >= "2" else "FastMCP (mcp 1.x)")
print("Done. If this was your first install, restart the kernel now.")

### Cell 2 — Set up Ollama (the free LLM)

**What it does:** gets a free AI model running inside Colab, in four small steps:

| Step | What happens | Time (Colab) |
|---|---|---|
| 1 | **Install** the Ollama program (skipped if already installed) | ~30 sec |
| 2 | **Start** the Ollama server in the background (it listens on `localhost:11434`) | ~5 sec |
| 3 | **Download** the model `llama3.2:3b` (about 2 GB, only the first time) | ~1–2 min |
| 4 | **Test** it: ask the model to say hello, so you know it works | ~5–30 sec |

> **What is a "server" here?** Ollama runs as a small background service. Our Python code sends
> it a question over `localhost` (meaning "this same computer") and gets an answer back.
> Don't confuse it with the *MCP server* we build in Cell 3 — that is a different program.

**Input:** nothing. (Optional: change `MODEL` to another tool-capable model such as
`"qwen2.5:3b"` or `"llama3.1:8b"`.)

**Output:** progress lines, then a greeting from the model:
```
STEP 1: Ollama is already installed (or: installing...)
STEP 2: Ollama server is running at http://localhost:11434
STEP 3: model 'llama3.2:3b' is ready (status: success)
STEP 4: model says -> Hello! It's nice to meet you today.
```

> **No GPU? It still works**, just more slowly. For speed: **Runtime → Change runtime type → T4 GPU**,
> then re-run from Cell 1.
>
> **Running on your own laptop instead of Colab?** Install Ollama first from <https://ollama.com/download>;
> this cell will detect it and skip step 1.

In [ ]:
# ---------- IMPORTS (each one explained) ----------
import shutil            # shutil.which("ollama") -> checks whether the Ollama program is already installed
import subprocess        # subprocess        -> lets Python run other programs (the installer, `ollama serve`)
import sys               # sys.platform      -> tells us which operating system we are on (Colab = "linux")
import time              # time.sleep        -> pause between checks while we wait for the server to start
import urllib.request    # urllib.request    -> make a simple web request to ask "are you awake?" (built in, no install)
import ollama            # ollama            -> the official Ollama library: download a model, chat with it

# The model we will use. "llama3.2:3b" = Meta's Llama 3.2, 3 billion parameters.
# Small enough for free Colab, and it supports TOOL CALLING (needed for MCP in Cell 5).
MODEL = "llama3.2:3b"

# The address where the Ollama server listens. "localhost" = this same computer. 11434 = Ollama's port.
OLLAMA_URL = "http://localhost:11434"


def ollama_is_running():
    """Return True if the Ollama server answers, otherwise False.
    Input : nothing.   Output: True / False."""
    try:
        urllib.request.urlopen(OLLAMA_URL, timeout=2)   # knock on the door; any answer = it is awake
        return True
    except Exception:                                   # no answer (or error) = not running yet
        return False


# ---------------------------------------------------------------- STEP 1: install Ollama
if shutil.which("ollama") is None:                      # None means "program not found on this computer"
    if sys.platform != "linux":                         # our automatic installer is for Colab (Linux) only
        raise RuntimeError("Please install Ollama from https://ollama.com/download and re-run this cell.")
    print("STEP 1: installing Ollama (about 30 seconds)...")
    # zstd is a tool the Ollama installer needs to unpack itself. -y = answer 'yes' automatically.
    subprocess.run("apt-get install -y -qq zstd", shell=True, capture_output=True)
    # Download the official install script and run it. (This is Ollama's documented install command.)
    result = subprocess.run("curl -fsSL https://ollama.com/install.sh | sh",
                            shell=True, capture_output=True, text=True)
    if result.returncode != 0:                          # non-zero = something went wrong -> show why
        print(result.stderr[-1500:])
        raise RuntimeError("Ollama install failed - see the message above.")
else:
    print("STEP 1: Ollama is already installed.")

# ---------------------------------------------------------------- STEP 2: start the Ollama server
if not ollama_is_running():
    # Send the server's messages to a real file (ollama_server.log) so they don't clutter the notebook
    # and so we can read them if something goes wrong.
    server_log = open("ollama_server.log", "w")
    # Popen = "start this program and keep going" (it keeps running in the background).
    subprocess.Popen(["ollama", "serve"], stdout=server_log, stderr=server_log)
    for _ in range(30):                                 # wait up to 30 seconds for it to wake up
        if ollama_is_running():
            break
        time.sleep(1)
    else:                                               # this 'else' runs only if the loop never 'break'-ed
        raise RuntimeError("Ollama server did not start. Check the file ollama_server.log")
print("STEP 2: Ollama server is running at", OLLAMA_URL)

# ---------------------------------------------------------------- STEP 3: download the model
# ollama.pull downloads the model (first time only; if you already have it, this is instant).
status = ollama.pull(MODEL).status
print(f"STEP 3: model '{MODEL}' is ready (status: {status})")

# ---------------------------------------------------------------- STEP 4: quick test
# ollama.chat sends a conversation to the model and returns its reply.
#   model    = which model to ask
#   messages = the conversation so far: a list of {"role": who is speaking, "content": what they said}
#   options  = settings; temperature 0 = "be as predictable as possible" (same question -> same answer)
reply = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "Say hello in five words."}],
    options={"temperature": 0},
)
print("STEP 4: model says ->", reply.message.content)

### Cell 3 — The MCP **server**

**What it does:** writes a small Python file to disk. `%%writefile` is a Jupyter magic that saves
the rest of the cell as a file instead of running it — we need a real file because the server runs
as a **separate process** (a separate program running alongside the notebook).

Read the file and notice how little there is to it:

* `@mcp.tool()` marks a normal Python function as callable by an AI.
* **The docstring (the text in triple quotes) is not a comment — it is the API description the LLM
  reads** to decide whether to call this tool. A vague docstring is a bug.
* The type hint `order_id: str` is what generates the JSON Schema in Cell 4. You write Python;
  MCP publishes a machine-readable contract.
* `mcp.run()` starts a loop that reads JSON from stdin and writes JSON to stdout. That is the
  entire transport.

**Input:** none (this cell only creates the file).

**Output:** `Writing orders_server.py`

After this, if you open the Colab file browser (📁 icon on the left) you will see `orders_server.py`.

---

#### The import at the top handles both SDK versions

The Python SDK renamed its main server class in **version 2.0**:

| SDK | Import | Class |
|---|---|---|
| `mcp` 1.x | `from mcp.server.fastmcp import FastMCP` | `FastMCP` |
| `mcp` 2.x | `from mcp.server.mcpserver import MCPServer` | `MCPServer` |

The `try` / `except ImportError` below means: *"try the new name; if that does not exist, use the old
one."* **Everything else is identical** — the protocol did not change, only one Python name did.

**In real life** this file would query your actual database. Nothing else about MCP changes.

In [ ]:
%%writefile orders_server.py
# ===== THIS WHOLE CELL IS SAVED AS A FILE called orders_server.py (it is not run right now) =====

# --- Import the server class, whichever SDK version is installed ---------------
# WHY: the `mcp` library gives us ready-made server code, so we don't write the protocol ourselves.
# mcp 2.0 renamed FastMCP -> MCPServer. Try the new name first, fall back to the old one.
try:
    from mcp.server.mcpserver import MCPServer as MCPServerClass   # mcp >= 2.0
except ImportError:
    from mcp.server.fastmcp import FastMCP as MCPServerClass       # mcp 1.x

# Create the server object. "orders" is its name (shown to the client during the handshake).
# log_level="ERROR" keeps the server quiet so only real problems are printed.
mcp = MCPServerClass("orders", log_level="ERROR")

# A stand-in for a real database: a Python dictionary of order-id -> order details.
ORDERS = {
    "A-1001": {"item": "Headphones", "amount_usd": 180, "status": "delivered"},
    "A-1002": {"item": "4K monitor", "amount_usd": 650, "status": "in transit"},
}

@mcp.tool()                                   # <- the magic line: publishes the function below over MCP
def lookup_order(order_id: str) -> str:
    """Look up a customer order by its ID, for example 'A-1001'."""
    # ^ THE LLM READS THIS SENTENCE to decide when to use the tool. Make it clear!
    #
    # Input : order_id  (text, e.g. "a-1002 ")     Output: one line of text describing the order

    key = order_id.strip().upper()            # tidy the input: remove spaces, make UPPERCASE ("a-1002 " -> "A-1002")
    if key not in ORDERS:                     # unknown order? return a helpful message instead of crashing
        return f"No order {key}. Known IDs: {', '.join(ORDERS)}"
    o = ORDERS[key]                           # fetch the order's details
    return f"{key}: {o['item']}, {o['amount_usd']} USD, {o['status']}"

# This block runs only when the file is started as a program (not when imported).
if __name__ == "__main__":
    mcp.run(transport="stdio")                # "stdio" = read JSON messages from stdin, write replies to stdout

### Cell 4 — The protocol, with nothing hiding it

**What it does:** starts the server as a subprocess and talks to it using **only `json` and
`subprocess`** — no MCP library on this side. This is the cell that makes MCP click, so read the
printed messages, not just the code.

**The transport:** one JSON object per line, into the server's stdin; one JSON object per line
back out of its stdout. That is it. (MCP calls this the *stdio* transport. There is also an HTTP
one for remote servers — same messages, different pipe.)

> **stdin / stdout in plain words:** every program has an "input pipe" (stdin) and an "output pipe"
> (stdout). We type into one end and read from the other — like passing notes through a tube.

**The conversation** is JSON-RPC 2.0 and has exactly three steps:

| # | Method | Meaning |
|---|---|---|
| 1 | `initialize` | "Hello, I speak version X." → server replies with who it is |
| 2 | `tools/list` | "What can you do?" → server replies with names, descriptions, **schemas** |
| 3 | `tools/call` | "Run `lookup_order` with these arguments" → server replies with the result |

Step 2 is the heart of MCP. The client did not know this server existed five seconds ago and
now has a machine-readable contract for it. **That is discovery, and it is why one client can
use any server.**

**Input:** none. (Cell 3 must have been run first so `orders_server.py` exists.)

**Output:** the three exchanges, then the tool's answer:
```
Server started with: /usr/bin/python3
1. initialize    -> connected to server 'orders'
2. tools/list    -> ['lookup_order']
     description : Look up a customer order by its ID, for example 'A-1001'.
     inputSchema : {'properties': {'order_id': {'title': 'Order Id', 'type': 'string'}}, ...}
3. tools/call    -> A-1002: 4K monitor, 650 USD, in transit
```

This cell also creates two things Cell 5 reuses: the list `TOOLS` and the function `mcp_call()`.

---

#### ⚠️ `RuntimeError: The MCP server exited instead of replying.`

It means **the server sent nothing back** — it died before it could answer. The error text printed
below it is the server's own complaint. Almost always one of these:

| What the server error says | Cause | Fix |
|---|---|---|
| `No module named 'mcp.server.fastmcp'. This is mcp 2.x ...` | SDK **2.x** installed but the server file uses the **1.x** import. | Re-run Cell 3 (it tries both names), then this cell. |
| `No module named 'mcp'` | **The wrong Python ran the server.** | The code below uses `sys.executable` (the exact Python running this notebook), which avoids this. |
| the `assert` message fires | Cell 3 was never run, so `orders_server.py` does not exist. | Run Cell 3. |
| A `SyntaxError` or other traceback | A typo inside the server file. | Fix it in Cell 3 and re-run Cell 3 before this cell. |

In [ ]:
# ---------- IMPORTS ----------
import json          # json       -> convert Python data <-> JSON text (the language MCP messages are written in)
import subprocess    # subprocess -> start the MCP server as a separate program and talk to it through pipes
import sys           # sys        -> gives us sys.executable = the exact Python that is running this notebook
import os            # os         -> lets us check whether a file exists

# Safety check: the server file must exist. If not, stop with a clear message. (Fix: run Cell 3.)
assert os.path.exists("orders_server.py"), "orders_server.py not found - run Cell 3 first."

# Keep the server's error messages in a file so we can SHOW them if the server dies.
# (Never throw them away: a silent server is impossible to debug.)
errlog = open("server_stderr.log", "w+")

# START THE SERVER as a child process and keep hold of its input/output pipes.
#   sys.executable        = the EXACT python running this notebook. (Writing "python3" instead is the
#                           #1 way to break an MCP server: it may be a different Python without `mcp`.)
#   stdin=PIPE/stdout=PIPE = we get a "tube" into the server and a "tube" out of it.
#   text=True             = send/receive normal text rather than raw bytes.
server = subprocess.Popen(
    [sys.executable, "orders_server.py"],
    stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=errlog,
    text=True, bufsize=1,
)


def send(method, params=None, msg_id=None):
    """Send ONE JSON-RPC message to the server and return its reply.

    Input : method  - what we ask for, e.g. "tools/list"
            params  - extra details for the request (a dictionary), optional
            msg_id  - a number. A message WITH an id expects a reply.
                      A message WITHOUT an id is a "notification": fire-and-forget, no reply.
    Output: the server's reply as a Python dictionary (or None for a notification).
    """
    msg = {"jsonrpc": "2.0", "method": method}       # every MCP message starts like this
    if params is not None:
        msg["params"] = params
    if msg_id is not None:
        msg["id"] = msg_id
    server.stdin.write(json.dumps(msg) + "\n")       # <-- THE ENTIRE "TRANSPORT": write one line of JSON
    server.stdin.flush()                             # push it through the pipe right now
    if msg_id is None:                               # notification: nothing to wait for
        return None
    line = server.stdout.readline()                  # wait for the server's one-line answer
    if not line:                                     # an empty answer means the server process died
        errlog.flush(); errlog.seek(0)
        raise RuntimeError("The MCP server exited instead of replying.\n"
                           "Its error output was:\n" + (errlog.read() or "(nothing)"))
    return json.loads(line)                          # turn the JSON text back into a Python dictionary


print("Server started with:", sys.executable)

# --- 1. HANDSHAKE: agree on a protocol version and say who we are ------------
# Input : our protocol version + our name.   Output: the server's name and abilities.
hello = send("initialize", {
    "protocolVersion": "2025-06-18",
    "capabilities": {},
    "clientInfo": {"name": "notebook-demo", "version": "1.0"},
}, msg_id=1)
send("notifications/initialized")                    # required "thanks, I'm ready" note (no reply expected)
print(f"1. initialize    -> connected to server '{hello['result']['serverInfo']['name']}'")

# --- 2. DISCOVERY: ask what this server can do -------------------------------
# Input : nothing.   Output: a list of tools; each has a name, description and input schema.
# We store it in TOOLS because Cell 5 will hand this list to the LLM.
TOOLS = send("tools/list", {}, msg_id=2)["result"]["tools"]
print(f"2. tools/list    -> {[t['name'] for t in TOOLS]}")
print(f"     description : {TOOLS[0]['description']}")
print(f"     inputSchema : {TOOLS[0]['inputSchema']}")


# --- 3. EXECUTION: actually run one -------------------------------------------
def mcp_call(name, args):
    """Ask the MCP server to run a tool and return its answer as text.
    Input : name = tool name, e.g. "lookup_order"      args = dictionary, e.g. {"order_id": "A-1002"}
    Output: text, e.g. "A-1002: 4K monitor, 650 USD, in transit"
    """
    reply = send("tools/call", {"name": name, "arguments": args}, msg_id=99)
    return reply["result"]["content"][0]["text"]     # dig the text out of the JSON reply


print(f"3. tools/call    -> {mcp_call('lookup_order', {'order_id': 'A-1002'})}")

### Cell 5 — End to end: the **Ollama LLM** decides, MCP executes

**What it does:** closes the loop. A human asks a question in English; the free Ollama model picks
the tool; we run it through MCP; the model writes the final answer.

**The punchline of the whole notebook is one line:**

```python
"parameters": t["inputSchema"]      # MCP's schema, handed straight to the LLM
```

MCP publishes each tool's schema in exactly the shape LLM tool-calling APIs already expect. So
"connecting an LLM to an MCP server" is not an integration project — it is a **rename**. That is
what makes an MCP server work in Claude Desktop, in an IDE, and in your own script without anyone
writing per-app glue.

**The hops** (watch them in the printout):

```
human question
   -> Ollama LLM  (given the tool list from Cell 4)
   -> LLM replies "call lookup_order(order_id='A-1002')"   <- it chose, we did not
   -> mcp_call()  -> tools/call -> MCP server -> result
   -> result fed back to the LLM
   -> LLM writes the human answer
```

> **Key idea:** the LLM never runs the tool itself. It only *asks*. **Our code** does the running.

**Input:** `MODEL` (from Cell 2), plus `TOOLS` and `mcp_call` (from Cell 4). No API key needed.
You can change the `question` line and re-run to ask something else
(try `"What did order A-1001 cost?"` or `"Where is order Z-9999?"`).

**Output:** (the exact wording of the last line will vary — LLMs are not word-for-word repeatable)
```
HUMAN     : Has my order A-1002 shipped yet?
LLM WANTS : lookup_order({'order_id': 'A-1002'})
MCP SAYS  : A-1002: 4K monitor, 650 USD, in transit
LLM ANSWER: Yes! Your 4K monitor (order A-1002) has shipped and is currently in transit.
```

> **If you see "No tool call"**: small free models sometimes answer without using the tool. Just
> re-run the cell, or change `MODEL` in Cell 2 to a bigger one (e.g. `"llama3.1:8b"` on a T4 GPU).

In [ ]:
# ---------- IMPORT ----------
import ollama     # ollama -> the library that lets Python chat with the free local LLM (already installed in Cell 1)

# ---- STEP A: describe our MCP tools in the format the LLM expects ---------------------------------
# This is THE ENTIRE "MCP -> LLM" INTEGRATION: we copy name, description and schema across.
# (MCP's "inputSchema" is already the right shape; the LLM calls it "parameters".)
#   Input : TOOLS (from Cell 4)       Output: `tools` - the same information in LLM format
tools = [{"type": "function",
          "function": {"name": t["name"],
                       "description": t["description"],       # the docstring from Cell 3!
                       "parameters": t["inputSchema"]}}       # the JSON Schema generated from the type hints
         for t in TOOLS]                                      # one entry per tool (we only have one)

# ---- STEP B: the conversation so far ---------------------------------------------------------------
question = "Has my order A-1002 shipped yet?"                 # <-- change me and re-run!
messages = [
    # "system" message = instructions for the model's behaviour (the human never sees it).
    {"role": "system", "content": "You are a helpful support agent. "
                                  "Use the provided tools to look up order information. "
                                  "Answer in one or two short sentences using ONLY what the tool returned."},
    # "user" message = what the human typed.
    {"role": "user", "content": question},
]
print("HUMAN     :", question)

# ---- HOP 1: show the LLM the question AND the tool list; the LLM decides what to do ----------------
#   Input : the model name, the conversation, the tool list.
#   Output: a message that EITHER contains tool_calls ("please run this tool") OR a plain text answer.
first = ollama.chat(model=MODEL, messages=messages, tools=tools,
                    options={"temperature": 0}).message       # temperature 0 = most predictable behaviour

if not first.tool_calls:                                      # the model chose NOT to use a tool
    print("LLM ANSWER:", first.content, "\n(No tool call - see the note above.)")
else:
    call = first.tool_calls[0]                                # take the first tool the LLM asked for
    name = call.function.name                                 # e.g. "lookup_order"
    args = dict(call.function.arguments)                      # e.g. {"order_id": "A-1002"}  (already a dictionary)
    print(f"LLM WANTS : {name}({args})")

    # ---- HOP 2: WE run the tool, through MCP (this is the Cell 4 function) --------------------------
    #   Input : tool name + arguments       Output: the text the server returned
    result = mcp_call(name, args)
    print("MCP SAYS  :", result)

    # ---- HOP 3: give the tool's result back to the LLM so it can answer the human -------------------
    # We add two messages to the conversation:
    #   1) the LLM's own request (so it remembers what it asked for)
    #   2) a "tool" message holding the result we got from MCP
    messages += [first, {"role": "tool", "tool_name": name, "content": result}]
    answer = ollama.chat(model=MODEL, messages=messages,
                         options={"temperature": 0}).message.content
    print("LLM ANSWER:", answer)

### Cell 6 — The same thing with the official SDK

**What it does:** everything Cell 4 did, in 8 lines. In real projects you write *this*; Cell 4
existed only so you know what it is doing underneath.

`stdio_client` spawns the server, `ClientSession` handles the handshake, and `list_tools()` /
`call_tool()` are the two methods you will use 95% of the time.

**Why `await` with no `asyncio.run()`?** Colab/Jupyter already runs an "event loop" (the machinery
that lets Python wait for things without freezing), so top-level `await` works in a cell — and
`asyncio.run()` would fail with *"this event loop is already running"*. In a plain `.py` script you
would wrap this in `async def main()` and call `asyncio.run(main())`.

> **One more v1 → v2 difference here.** SDK 2.0 switched result fields from camelCase to
> snake_case, so `info.serverInfo` became `info.server_info`. The line below reads whichever one
> exists. The imports themselves are unchanged across both versions.

**Input:** none.

**Output:**
```
server : orders
tools  : ['lookup_order']
result : A-1001: Headphones, 180 USD, delivered
```

---

#### ⚠️ `UnsupportedOperation: fileno` in Colab or Jupyter

**This one is a notebook problem, not an MCP problem.** `stdio_client` starts the server as a
subprocess and, by default, sends the server's error messages to *your* notebook's error stream.
A subprocess needs a real operating-system file for that, and a notebook's error stream is not a
real file. The fix is to give `stdio_client` a real file to write to:

```python
sdk_errlog = open("sdk_server_stderr.log", "w")
async with stdio_client(params, errlog=sdk_errlog) as (read, write):
```

That is also where the server's crash messages land, which is handy when debugging.

In [ ]:
import warnings; warnings.filterwarnings("ignore")   # hide a harmless SDK warning so the output stays clean

# ---------- IMPORTS ----------
# ClientSession        -> the ready-made MCP CLIENT: does the handshake, list_tools, call_tool for us.
# StdioServerParameters-> a small "how to start the server" description (which program, which file).
# stdio_client         -> starts the server program and connects pipes to it (the stdio transport from Cell 4).
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import sys                                             # sys.executable = the Python running this notebook

# Tell the SDK HOW to start our server: "run  <this python>  orders_server.py"
params = StdioServerParameters(command=sys.executable, args=["orders_server.py"])

# The server's error output must go to a REAL file (see the note above about "fileno").
sdk_errlog = open("sdk_server_stderr.log", "w")

# `async with` = "start it, use it, and shut it down cleanly when the block ends".
async with stdio_client(params, errlog=sdk_errlog) as (read, write):   # start server, get the two pipes
    async with ClientSession(read, write) as session:                   # wrap the pipes in a client session
        info = await session.initialize()               # HANDSHAKE (same as Cell 4, step 1)

        # mcp 1.x calls this field serverInfo; mcp 2.x renamed it to server_info. Read whichever exists.
        server_info = getattr(info, "server_info", None) or getattr(info, "serverInfo")
        print("server :", server_info.name)

        tools = await session.list_tools()              # DISCOVERY (same as Cell 4, step 2)
        print("tools  :", [t.name for t in tools.tools])

        # EXECUTION (same as Cell 4, step 3). Input: tool name + arguments. Output: result object.
        out = await session.call_tool("lookup_order", {"order_id": "A-1001"})
        print("result :", out.content[0].text)          # the text lives in out.content[0].text

# If the server complained about anything, show it rather than leaving it hidden in the file.
sdk_errlog.close()
complaints = open("sdk_server_stderr.log").read().strip()
if complaints:
    print("\nserver stderr:\n" + complaints)

# Tidy up: switch off the raw server we started in Cell 4 (skipped if you did not run Cell 4).
if "server" in globals():
    server.terminate()
    print("\nRaw server from Cell 4 closed.")

---

## Recap

```
        YOU                    HOST (AI app)              SERVER (your code)
         |                          |                            |
   "Has A-1002        (1) initialize  ------------------------->  |
    shipped?"   --->  (2) tools/list  ------------------------->  |
                          <---- name + description + schema ----  |
                          |                                      |
                      Ollama LLM reads the schema and chooses    |
                          |                                      |
                      (3) tools/call {order_id: "A-1002"} ---->  lookup_order()
                          <---- "A-1002: 4K monitor, in transit"  |
                          |                                      |
   answer  <----  LLM turns that into a sentence                  |
```

### The five things worth remembering

1. **MCP is JSON-RPC over a pipe.** Cell 4 used no MCP library and worked. Nothing is hidden.
2. **Three methods carry almost everything:** `initialize`, `tools/list`, `tools/call`.
3. **`tools/list` is the point.** Discovery at runtime is why one client works with any server —
   the *N × M* problem becomes *N + M*.
4. **The docstring and type hints are the API.** They become the description and JSON Schema the
   model reads. Sloppy ones produce an agent that calls the wrong tool with the wrong arguments.
5. **The LLM never touches your data.** It only ever says *"I would like to call this"*. Your
   client decides whether to obey — which is where you put permissions, limits and audit logging.

### What we left out (deliberately)

Beyond tools, a server can also expose **resources** (readable content, like files) and
**prompts** (reusable templates). Servers can run remotely over HTTP instead of stdio, and can
require OAuth. Same three-step conversation underneath.

### Try it

1. Add a second tool (`cancel_order`) to Cell 3 and re-run Cells 3–5. Notice you change
   **nothing** in the client — discovery picks it up.
2. Change the docstring to something vague like `"order stuff"` and watch the LLM's choice get worse.
3. Swap the model: change `MODEL` in Cell 2 to `"qwen2.5:3b"` or `"llama3.1:8b"` and compare answers.
   (Only models that support **tools** work. See <https://ollama.com/search?c=tools>.)
4. Point a real MCP host at this server. For Claude Desktop, add to its config file:
   ```json
   { "mcpServers": { "orders": {
       "command": "/full/path/to/the/same/python",
       "args": ["/full/path/to/orders_server.py"] } } }
   ```
   Same file, no changes — that is the whole promise of MCP.

### Reference
* Specification — <https://modelcontextprotocol.io/specification>
* Python SDK — <https://github.com/modelcontextprotocol/python-sdk>
* Server directory — <https://github.com/modelcontextprotocol/servers>
* Ollama — <https://ollama.com> · Ollama Python library — <https://github.com/ollama/ollama-python>